In [1]:

## imports

import os
import sys
from pathlib import Path

import yaml
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Make the project root importable in a notebook context
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag.embedding_models.instance import embedding_manager
from src.rag.vector_store.instance import vector_store



C:\Users\gabri\AppData\Local\Temp\ipykernel_65468\794946790.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
currentDirectory = os.getcwd()
currentDirectory

'c:\\Users\\gabri\\Documents\\capstone-projects\\airline-support-bot\\backend\\rag-service\\notebooks'

In [3]:
## load data from the knowledge base 
loader = DirectoryLoader("../data/raw/",glob="**/*.md",loader_cls=TextLoader)
documents = loader.load()

print(f"number of documents :{len(documents)}")


number of documents :30


In [4]:
print(f"document one:{documents[0]}")
print("Content:\n", documents[0].page_content)
print("Metadata:\n", documents[0].metadata)


document one:page_content='---
document_id: KQ-AIRPORT-001
title: Airport Lounge Services
origin: KENYA AIRWAYS
domain: AIRPORT_SERVICES
category: LOUNGE_SERVICES
document_type: POLICY

applicable_to:
  - CUSTOMER
  - CUSTOMER_SERVICE_AGENT

access: PUBLIC
status: ACTIVE

language: EN
---

# Airport Lounge Services

Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.

## Kenya Airways Lounges

Kenya Airways operates the following lounges:

- Pride Lounge
- Simba Lounge
- Asante Lounge
- Msafiri Lounge

## Lounge Access Eligibility

Passengers eligible for Kenya Airways lounge access include:

- Kenya Airways Business Class passengers.
- SkyTeam Platinum and Gold cardholders.
- Eligible passengers travelling on Kenya Airw

In [5]:
## adding extra metadata to the documents
for document in documents:
    content = document.page_content

    # Split the front matter from the Markdown content
    if content.startswith("---"):
        _, front_matter, markdown = content.split("---", 2)

        # Convert YAML front matter into a Python dictionary
        metadata = yaml.safe_load(front_matter)

        # Add your metadata to LangChain's existing metadata
        document.metadata.update(metadata)

        # Remove the metadata from the actual page content
        document.page_content = markdown

In [6]:
## creating chunks from the documents
## 1. Creating the text splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "section"),
        ("##", "subsection"),
        ("###", "subsubsection")
    ]
)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [7]:

chunks = []

for document in documents:

    # First split according to Markdown headings
    sections = markdown_splitter.split_text(
        document.page_content
    )

    for section in sections:

        # Carry the original document metadata into the section
        section.metadata.update(document.metadata)

        # Split the section further if it is too large
        smaller_chunks = recursive_splitter.split_documents(
            [section]
        )

        for chunk in smaller_chunks:

            # Give every final chunk a unique ID
            chunk.metadata["chunk_id"] = (
                f"{document.metadata.get('document_id', 'UNKNOWN')}"
                f"-chunk-{len(chunks) + 1:03d}"
            )

            chunks.append(chunk)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 279


In [8]:
for chunk in chunks:
    print(f"Chunk ID: {chunk.metadata['chunk_id']}")
    print(f"Chunk Content: {chunk.page_content}")  # Print first 100 characters
    print(f"Chunk Metadata: {chunk.metadata}")
    


Chunk ID: KQ-AIRPORT-001-chunk-001
Chunk Content: Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.
Chunk Metadata: {'section': 'Airport Lounge Services', 'source': '..\\data\\raw\\airport_services\\lounge_services.md', 'document_id': 'KQ-AIRPORT-001', 'title': 'Airport Lounge Services', 'origin': 'KENYA AIRWAYS', 'domain': 'AIRPORT_SERVICES', 'category': 'LOUNGE_SERVICES', 'document_type': 'POLICY', 'applicable_to': ['CUSTOMER', 'CUSTOMER_SERVICE_AGENT'], 'access': 'PUBLIC', 'status': 'ACTIVE', 'language': 'EN', 'chunk_id': 'KQ-AIRPORT-001-chunk-001'}
Chunk ID: KQ-AIRPORT-001-chunk-002
Chunk Content: Kenya Airways operates the following lounges:  
- Pride Lounge
- Simba Lounge
- Asante Lounge
- Msafiri Lounge
Chunk Me

In [ ]:

embeddings = embedding_manager.create_embeddings(chunks)

vector_store.add_chunks(
    chunks,
    embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]